# 🏥 ICD-Insight — Notebook B: Inference on New Discharge Summaries
### Upload a PDF → Get ICD-10 Codes + PDF Report

> **M.Tech Final Year Project** | ICD-10 Multi-Label Classification  
> Run on **Google Colab** (GPU recommended for speed, but CPU works too)

---

**What this notebook does:**
1. Installs dependencies
2. Loads trained LoRA adapters from GitHub
3. **Prompts you to upload a discharge summary PDF**
4. Extracts text and removes PHI
5. Predicts ICD-10-CM codes with confidence scores
6. Extracts evidence snippets per code
7. **Generates and downloads a clean PDF report**

**Quick start:**  
Run all cells in order (Ctrl+F9 or Runtime → Run All).  
Cell B.3 will pause and show a file picker — upload your PDF there.

## B.0 — Install Dependencies

> ⏱ Takes ~3–5 minutes on first run.

In [ ]:
%%capture
!pip install -q \
    transformers==4.41.0 \
    peft==0.11.0 \
    bitsandbytes==0.43.1 \
    accelerate==0.30.0 \
    pymupdf==1.24.3 \
    presidio-analyzer==2.2.354 \
    presidio-anonymizer==2.2.354 \
    reportlab==4.2.0 \
    spacy==3.7.4

!python -m spacy download en_core_web_lg -q

print("✅ All dependencies installed")

In [ ]:
import torch, transformers, peft, fitz, reportlab
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"CUDA avail   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

## B.1 — Load Trained Adapters from GitHub

The LoRA adapter weights (~20–50 MB) are stored in the GitHub repo under `checkpoints/`.

In [ ]:
# ─── CONFIGURATION — Edit GITHUB_REPO if your repo URL differs ───────────────

GITHUB_REPO     = "https://github.com/YOUR_USERNAME/icd-insight.git"  # ← update this
BASE_MODEL_ID   = "NLP4Science/BioClinical-ModernBERT-base"
MAX_LENGTH      = 2048
THRESHOLD       = 0.5    # sigmoid threshold — codes above this are predicted
TOP_N_CODES     = 15     # max codes to show in report
STRATEGY        = "qlora"  # "qlora" or "lora"

REPO_DIR        = "/content/icd-insight"
ADAPTER_DIR     = f"{REPO_DIR}/checkpoints"
ICD_DESC_TSV    = f"{REPO_DIR}/data/processed/icd10_descriptions.tsv"

print(f"Repo         : {GITHUB_REPO}")
print(f"Adapter dir  : {ADAPTER_DIR}")
print(f"Threshold    : {THRESHOLD}")
print(f"Max tokens   : {MAX_LENGTH}")

In [ ]:
import os, subprocess

if os.path.exists(REPO_DIR):
    print("▶ Repo already cloned — pulling latest changes …")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"▶ Cloning {GITHUB_REPO} …")
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)

# Check adapter files exist
required_files = [
    f"{ADAPTER_DIR}/adapter_config.json",
    f"{ADAPTER_DIR}/label_binarizer.pkl",
]

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌ MISSING"
    print(f"  {status}  {os.path.basename(f)}")
    if not exists:
        all_ok = False

# Check safetensors (might be in LFS)
adapter_weights = f"{ADAPTER_DIR}/adapter_model.safetensors"
if os.path.exists(adapter_weights):
    size_mb = os.path.getsize(adapter_weights) / 1e6
    print(f"  ✅  adapter_model.safetensors ({size_mb:.1f} MB)")
else:
    print("  ⚠  adapter_model.safetensors not found — check Git LFS")
    print("     Run: !git lfs pull  (if your repo uses Git LFS)")

if not all_ok:
    print("\n⚠ Some files are missing. Run Notebook A first to train and upload adapters.")
else:
    print("\n✅ Adapter files verified — ready to load model")

## B.2 — Load Base Model + LoRA Adapters

In [ ]:
import pickle, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Load label binarizer
with open(f"{ADAPTER_DIR}/label_binarizer.pkl", "rb") as f:
    MLB = pickle.load(f)

NUM_LABELS = len(MLB.classes_)
print(f"  Label classes : {NUM_LABELS} ICD-10 codes")

# Load base model
print(f"\n▶ Loading {BASE_MODEL_ID} ({STRATEGY.upper()}) …")

if STRATEGY == "qlora":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_ID,
        num_labels=NUM_LABELS,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        problem_type="multi_label_classification",
    )
else:  # lora fallback
    base_model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_ID,
        num_labels=NUM_LABELS,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        problem_type="multi_label_classification",
    )

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("  LoRA adapter loaded ✅")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
print(f"  Tokenizer loaded ✅")

DEVICE = next(model.parameters()).device
print(f"  Device: {DEVICE}")

if torch.cuda.is_available():
    used_gb = torch.cuda.memory_allocated() / 1e9
    print(f"  VRAM used: {used_gb:.2f} GB")

In [ ]:
import csv

def load_icd_descriptions(tsv_path):
    """Load ICD-10-CM code descriptions from TSV file."""
    desc = {}
    if os.path.exists(tsv_path):
        with open(tsv_path, newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f, delimiter="\t"):
                code = row.get("icd_code", "").strip()
                description = row.get("description", "").strip()
                if code:
                    desc[code] = description
        print(f"  Loaded {len(desc):,} ICD-10 descriptions from {tsv_path}")
    else:
        print(f"  ⚠ Description file not found — codes will display without descriptions")
    return desc

ICD_DESCRIPTIONS = load_icd_descriptions(ICD_DESC_TSV)

## B.3 — Upload Discharge Summary PDF

> **👆 Run this cell and use the file picker that appears to upload your discharge summary PDF.**  
> The file will be processed automatically in the next cells.

In [ ]:
from google.colab import files

print("📁 Please upload your discharge summary PDF file …")
print("   (Supported format: .pdf only)")
print()

uploaded = files.upload()

# Validate upload
pdf_files = [name for name in uploaded.keys() if name.lower().endswith(".pdf")]

if not pdf_files:
    raise ValueError("❌ No PDF file uploaded. Please re-run this cell and upload a .pdf file.")

UPLOADED_PDF = f"/content/{pdf_files[0]}"
print(f"\n✅ File received: {pdf_files[0]}")
size_kb = os.path.getsize(UPLOADED_PDF) / 1024
print(f"   Size: {size_kb:.1f} KB")

## B.4 — PDF Parsing & PHI Removal

In [ ]:
import fitz  # PyMuPDF
import re

def extract_text_from_pdf(pdf_path):
    """Extract all text from PDF using PyMuPDF."""
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    doc.close()
    return "\n".join(pages), len(pages)

def extract_patient_info(text):
    """Heuristically extract patient name, ID, and dates."""
    name = "Unknown"
    patient_id = "Unknown"
    admission_date = ""
    discharge_date = ""

    name_m = re.search(r"(?:patient[\s_]?name|name)\s*[:\-]\s*([A-Za-z ,'-]+)", text, re.I)
    if name_m: name = name_m.group(1).strip().title()

    id_m = re.search(r"(?:mrn|patient[\s_]?id|id)\s*[:\-#]\s*([A-Za-z0-9\-]+)", text, re.I)
    if id_m: patient_id = id_m.group(1).strip()

    adm_m = re.search(r"(?:admission|admitted).*?(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4})", text, re.I)
    if adm_m: admission_date = adm_m.group(1)

    dis_m = re.search(r"(?:discharge).*?(\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4})", text, re.I)
    if dis_m: discharge_date = dis_m.group(1)

    return name, patient_id, admission_date, discharge_date

print(f"▶ Extracting text from {os.path.basename(UPLOADED_PDF)} …")
raw_text, page_count = extract_text_from_pdf(UPLOADED_PDF)
print(f"  Pages     : {page_count}")
print(f"  Characters: {len(raw_text):,}")
print(f"  Words     : {len(raw_text.split()):,}")

patient_name, patient_id, admission_date, discharge_date = extract_patient_info(raw_text)
print(f"\n  Patient Name    : {patient_name}")
print(f"  Patient ID/MRN  : {patient_id}")
print(f"  Admission Date  : {admission_date or '—'}")
print(f"  Discharge Date  : {discharge_date or '—'}")
print()
print("  First 500 chars:")
print("  " + raw_text[:500].replace("\n", " "))

In [ ]:
# PHI Removal using Microsoft Presidio
# If Presidio is unavailable, falls back to regex-based cleaning

def anonymise_phi_presidio(text):
    """Remove PHI using Presidio NER-based anonymisation."""
    try:
        from presidio_analyzer import AnalyzerEngine
        from presidio_anonymizer import AnonymizerEngine
        from presidio_anonymizer.entities import OperatorConfig

        analyzer   = AnalyzerEngine()
        anonymizer = AnonymizerEngine()

        entities = ["PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS",
                    "US_SSN", "LOCATION", "MEDICAL_LICENSE", "URL"]

        results = analyzer.analyze(text=text, entities=entities, language="en")
        anon    = anonymizer.anonymize(
            text=text, analyzer_results=results,
            operators={"DEFAULT": OperatorConfig("replace", {"new_value": "[REDACTED]"})},
        )
        return anon.text, "presidio"
    except Exception as e:
        print(f"  Presidio fallback: {e}")
        return text, "none"

print("▶ Applying PHI anonymisation …")
clean_text, method = anonymise_phi_presidio(raw_text)

# Basic cleanup
clean_text = re.sub(r"\s+", " ", clean_text).strip()

redacted_count = clean_text.count("[REDACTED]")
print(f"  Method        : {method}")
print(f"  PHI redacted  : {redacted_count} entities")
print(f"  Clean text len: {len(clean_text):,} chars")

# Token count estimate
approx_tokens = len(clean_text.split()) * 1.3
print(f"  Est. tokens   : ~{int(approx_tokens):,} (will truncate to {MAX_LENGTH})")

## B.5 — ICD-10 Inference

In [ ]:
import numpy as np

print(f"▶ Tokenizing (max_length={MAX_LENGTH}) …")
encoding = tokenizer(
    clean_text,
    max_length=MAX_LENGTH,
    truncation=True,
    padding="max_length",
    return_tensors="pt",
)

input_ids      = encoding["input_ids"].to(DEVICE)
attention_mask = encoding["attention_mask"].to(DEVICE)

actual_tokens  = (attention_mask == 1).sum().item()
was_truncated  = actual_tokens == MAX_LENGTH
print(f"  Tokens used: {actual_tokens:,} {'(TRUNCATED)' if was_truncated else ''}")

print("▶ Running inference …")
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits  = outputs.logits.squeeze(0).cpu()
    probs   = torch.sigmoid(logits).numpy()

# Apply threshold
predicted_indices = np.where(probs >= THRESHOLD)[0]
predicted_indices = sorted(predicted_indices, key=lambda i: -probs[i])[:TOP_N_CODES]

predictions = []
for idx in predicted_indices:
    code = MLB.classes_[idx]
    conf = float(probs[idx])
    desc = ICD_DESCRIPTIONS.get(code, f"ICD-10-CM: {code}")
    predictions.append({"icd_code": code, "description": desc,
                        "confidence": round(conf, 4), "evidence": []})

print(f"\n✅ Predicted {len(predictions)} ICD-10 codes (threshold={THRESHOLD})")
print()
print(f"{'Code':<12} {'Confidence':<12} {'Description'}")
print("-" * 70)
for p in predictions:
    bar = "█" * int(p['confidence'] * 20) + "░" * (20 - int(p['confidence'] * 20))
    print(f"{p['icd_code']:<12} {p['confidence']:.1%}  {bar}  {p['description'][:35]}")

## B.6 — Evidence Extraction (XAI)

Uses attention weights to identify which text spans triggered each prediction.  
This makes the model interpretable and clinically useful.

In [ ]:
def extract_evidence_for_code(clean_text, tokenizer, model, input_ids, attention_mask,
                               label_idx, top_k=3, window=100):
    """Extract top-k text snippets as evidence using attention attribution."""
    try:
        with torch.no_grad():
            out = model(input_ids=input_ids, attention_mask=attention_mask,
                        output_attentions=True)

        if not out.attentions:
            return []

        # Average attention across layers and heads → CLS row
        attn_stack = torch.stack(out.attentions)  # [L, 1, H, S, S]
        attn_mean  = attn_stack.mean(dim=0).mean(dim=1).squeeze(0)  # [S, S]
        cls_attn   = attn_mean[0, :].cpu().numpy()  # [S]

        tokens     = tokenizer.convert_ids_to_tokens(input_ids.squeeze().tolist())
        top_pos    = np.argsort(cls_attn)[::-1][:top_k * 8]

        snippets   = []
        seen       = set()

        for pos in top_pos:
            if pos == 0 or pos >= len(tokens): continue
            tok = tokens[pos]
            if tok in ["[CLS]", "[SEP]", "[PAD]", "<s>", "</s>", "<pad>"]: continue

            prefix_str  = tokenizer.convert_tokens_to_string(tokens[1:pos])
            char_start  = max(0, len(prefix_str) - window // 2)
            char_end    = min(len(clean_text), char_start + window)
            snippet     = clean_text[char_start:char_end].strip()

            # Deduplicate
            snippet_key = snippet[:40]
            if len(snippet) > 20 and snippet_key not in seen:
                seen.add(snippet_key)
                snippets.append(snippet)

            if len(snippets) >= top_k:
                break

        return snippets
    except Exception:
        return []


print("▶ Extracting evidence snippets for each predicted code …")
print("  (This may take 30–60 seconds)")

for i, pred in enumerate(predictions):
    label_idx = list(MLB.classes_).index(pred["icd_code"])
    evidence  = extract_evidence_for_code(
        clean_text, tokenizer, model, input_ids, attention_mask,
        label_idx=label_idx, top_k=3, window=100
    )
    pred["evidence"] = evidence
    print(f"  [{i+1:2d}/{len(predictions)}] {pred['icd_code']:10s} — {len(evidence)} snippet(s)")

print("\n✅ Evidence extraction complete")

# Preview first prediction
if predictions:
    p = predictions[0]
    print(f"\nExample — {p['icd_code']} ({p['confidence']:.1%}):")
    for j, ev in enumerate(p['evidence'][:2], 1):
        print(f'  Evidence {j}: "{ev[:120]}"')

## B.7 — Generate PDF Report

Creates a structured, professional PDF report using ReportLab with:  
- Patient info header  
- Colour-coded ICD-10 code table with confidence bars  
- Evidence snippets per code  
- Clinical disclaimer

In [ ]:
from datetime import datetime
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    HRFlowable
)

# Colour palette
DARK_BLUE   = colors.HexColor("#1B3A5C")
MED_BLUE    = colors.HexColor("#2E6DA4")
LIGHT_BLUE  = colors.HexColor("#D6E8F7")
TEAL        = colors.HexColor("#17A58D")
AMBER       = colors.HexColor("#E8A020")
LIGHT_GREY  = colors.HexColor("#F4F6F8")
MID_GREY    = colors.HexColor("#8C9BAB")
RED_LIGHT   = colors.HexColor("#FDE8E8")

def make_style(name, **kwargs):
    return ParagraphStyle(name, **kwargs)

styles = {
    "title":    make_style("title",    fontSize=18, textColor=DARK_BLUE, fontName="Helvetica-Bold"),
    "body":     make_style("body",     fontSize=9,  textColor=colors.HexColor("#2C3E50"), fontName="Helvetica", leading=13),
    "section":  make_style("section",  fontSize=12, textColor=DARK_BLUE, fontName="Helvetica-Bold", spaceBefore=12, spaceAfter=4),
    "evidence": make_style("evidence", fontSize=8.5, textColor=colors.HexColor("#2C3E50"), fontName="Helvetica-Oblique", leading=12, leftIndent=12),
    "small":    make_style("small",    fontSize=8,  textColor=MID_GREY,  fontName="Helvetica"),
    "disc":     make_style("disc",     fontSize=8,  textColor=colors.HexColor("#C0392B"), fontName="Helvetica-Bold"),
    "bar":      make_style("bar",      fontSize=7,  fontName="Helvetica", leading=10),
}


def page_decoration(canvas, doc):
    canvas.saveState()
    w, h = A4
    # Header
    canvas.setFillColor(DARK_BLUE)
    canvas.rect(0, h - 22*mm, w, 22*mm, fill=True, stroke=False)
    canvas.setFillColor(colors.white)
    canvas.setFont("Helvetica-Bold", 11)
    canvas.drawString(15*mm, h - 13*mm, "ICD-Insight  ·  Clinical ICD-10 Prediction Report")
    canvas.setFont("Helvetica", 9)
    canvas.drawRightString(w - 15*mm, h - 13*mm, "CONFIDENTIAL — FOR CLINICAL USE ONLY")
    # Footer
    canvas.setFillColor(LIGHT_GREY)
    canvas.rect(0, 0, w, 12*mm, fill=True, stroke=False)
    canvas.setFillColor(MID_GREY)
    canvas.setFont("Helvetica", 8)
    canvas.drawString(15*mm, 4*mm, f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}  ·  ICD-Insight M.Tech Project")
    canvas.drawRightString(w - 15*mm, 4*mm, f"Page {doc.page}")
    canvas.restoreState()


# Build report filename
safe_id     = patient_id.replace(" ", "_")[:20]
OUTPUT_PDF  = f"/content/icd10_report_{safe_id}.pdf"

doc   = SimpleDocTemplate(OUTPUT_PDF, pagesize=A4,
                           leftMargin=15*mm, rightMargin=15*mm,
                           topMargin=30*mm,  bottomMargin=20*mm)
story = []

# Title + patient info
story.append(Paragraph("Clinical ICD-10 Prediction Report", styles["title"]))
story.append(Spacer(1, 3*mm))

info_data = [
    ["Patient Name", patient_name, "Patient ID / MRN", patient_id],
    ["Admission Date", admission_date or "—", "Discharge Date", discharge_date or "—"],
]
info_tbl = Table(info_data, colWidths=[38*mm, 55*mm, 38*mm, 50*mm])
info_tbl.setStyle(TableStyle([
    ("BACKGROUND",  (0,0), (-1,-1), LIGHT_GREY),
    ("BACKGROUND",  (0,0), (0,-1), LIGHT_BLUE), ("BACKGROUND", (2,0), (2,-1), LIGHT_BLUE),
    ("FONTNAME",    (0,0), (0,-1), "Helvetica-Bold"), ("FONTNAME", (2,0), (2,-1), "Helvetica-Bold"),
    ("FONTSIZE",    (0,0), (-1,-1), 9),
    ("TEXTCOLOR",   (0,0), (0,-1), DARK_BLUE), ("TEXTCOLOR", (2,0), (2,-1), DARK_BLUE),
    ("GRID",        (0,0), (-1,-1), 0.5, colors.HexColor("#C5D5E8")),
    ("TOPPADDING",  (0,0), (-1,-1), 5), ("BOTTOMPADDING", (0,0), (-1,-1), 5),
    ("LEFTPADDING", (0,0), (-1,-1), 8),
]))
story.append(info_tbl)
story.append(Spacer(1, 5*mm))
story.append(HRFlowable(width="100%", thickness=1.5, color=MED_BLUE))
story.append(Spacer(1, 4*mm))

# ICD code table
story.append(Paragraph("Predicted ICD-10-CM Codes", styles["section"]))
tbl_data = [["ICD-10 Code", "Description", "Confidence", "Confidence Bar"]]

for pred in predictions:
    conf  = pred["confidence"]
    c     = TEAL if conf >= 0.75 else (MED_BLUE if conf >= 0.50 else AMBER)
    chex  = c.hexval()[2:]
    bar   = "█" * int(conf * 20) + "░" * (20 - int(conf * 20))
    tbl_data.append([
        Paragraph(f"<b>{pred['icd_code']}</b>", styles["body"]),
        Paragraph(pred["description"] or "", styles["body"]),
        Paragraph(f'<font color="#{chex}">{conf:.1%}</font>', styles["body"]),
        Paragraph(f'<font color="#{chex}">{bar}</font>', styles["bar"]),
    ])

code_tbl = Table(tbl_data, colWidths=[28*mm, 80*mm, 22*mm, 47*mm], repeatRows=1)
code_tbl.setStyle(TableStyle([
    ("BACKGROUND",    (0,0), (-1,0), DARK_BLUE), ("TEXTCOLOR", (0,0), (-1,0), colors.white),
    ("FONTNAME",      (0,0), (-1,0), "Helvetica-Bold"), ("FONTSIZE", (0,0), (-1,0), 9),
    ("ROWBACKGROUNDS",(0,1), (-1,-1), [colors.white, LIGHT_GREY]),
    ("GRID",          (0,0), (-1,-1), 0.4, colors.HexColor("#C5D5E8")),
    ("TOPPADDING",    (0,0), (-1,-1), 5), ("BOTTOMPADDING", (0,0), (-1,-1), 5),
    ("LEFTPADDING",   (0,0), (-1,-1), 6), ("VALIGN", (0,0), (-1,-1), "MIDDLE"),
]))
story.append(code_tbl)
story.append(Spacer(1, 6*mm))

# Evidence
story.append(HRFlowable(width="100%", thickness=1, color=LIGHT_BLUE))
story.append(Paragraph("Evidence Snippets (per code)", styles["section"]))

for pred in predictions:
    if not pred.get("evidence"): continue
    story.append(Paragraph(f"<b>{pred['icd_code']}</b> — {pred['description']}", styles["body"]))
    for i, snip in enumerate(pred["evidence"][:3], 1):
        story.append(Paragraph(f'{i}. "{snip.strip()}"', styles["evidence"]))
    story.append(Spacer(1, 3*mm))

# Model info
story.append(HRFlowable(width="100%", thickness=0.5, color=MID_GREY))
story.append(Spacer(1, 3*mm))
story.append(Paragraph(
    f"Model: {BASE_MODEL_ID} + QLoRA  ·  Threshold: {THRESHOLD}  ·  Dataset: birgermoell/icd10-clinical-notes (Public HF) (top-{len(MLB.classes_)} ICD-10-CM)",
    styles["small"]))
story.append(Spacer(1, 4*mm))

# Disclaimer
disc_text = (
    "⚠ DISCLAIMER: This report is AI-generated for research purposes only. "
    "It is NOT a substitute for clinical coding by a certified medical coder or physician. "
    "All codes must be reviewed and verified by qualified clinical staff before use."
)
disc_tbl = Table([[Paragraph(disc_text, styles["disc"])]], colWidths=[177*mm])
disc_tbl.setStyle(TableStyle([
    ("BACKGROUND",    (0,0), (-1,-1), RED_LIGHT),
    ("BOX",           (0,0), (-1,-1), 1, colors.HexColor("#C0392B")),
    ("TOPPADDING",    (0,0), (-1,-1), 6), ("BOTTOMPADDING", (0,0), (-1,-1), 6),
    ("LEFTPADDING",   (0,0), (-1,-1), 10),
]))
story.append(disc_tbl)

doc.build(story, onFirstPage=page_decoration, onLaterPages=page_decoration)
size_kb = os.path.getsize(OUTPUT_PDF) / 1024
print(f"\n✅ Report generated: {os.path.basename(OUTPUT_PDF)} ({size_kb:.1f} KB)")

## B.8 — Download Report

Your ICD-10 prediction report will auto-download to your browser.

In [ ]:
from google.colab import files
from IPython.display import display, HTML

# Display inline summary first
print("=" * 60)
print("INFERENCE SUMMARY")
print("=" * 60)
print(f"Patient        : {patient_name}")
print(f"Patient ID     : {patient_id}")
print(f"Codes predicted: {len(predictions)}")
print()
print(f"{'Code':<12} {'Confidence':<12} Description")
print("-" * 60)
for p in predictions:
    print(f"{p['icd_code']:<12} {p['confidence']:.1%}       {p['description'][:35]}")

print()
print("📄 Downloading PDF report …")
files.download(OUTPUT_PDF)

print()
print("✅ Done! Report downloaded to your computer.")
print(f"   File: {os.path.basename(OUTPUT_PDF)}")

---
## 📋 Inference Notebook Summary

| Step | Description |
|------|-------------|
| B.1 | LoRA adapters loaded from GitHub |
| B.2 | BioClinical ModernBERT loaded in 4-bit (QLoRA) |
| B.3 | Discharge summary PDF uploaded by user |
| B.4 | Text extracted, PHI anonymised via Presidio |
| B.5 | ICD-10 codes predicted with confidence scores |
| B.6 | Evidence snippets extracted via attention attribution |
| B.7 | PDF report generated (ReportLab) |
| B.8 | Report downloaded to user's machine |

**To run for a new patient:** restart the notebook and run all cells again (B.3 will prompt for a new PDF upload).

---
*ICD-Insight — M.Tech Final Year Project*